### 1. Chargement du modèle et vérification des données 

Je charge spaCy pour faire de la NER en français.  Le but est de m’assurer que la structure est OK et qu’on est bien dans la fourchette attendue.

In [1]:
import spacy
nlp = spacy.load("fr_core_news_sm")
print("Modèle spaCy chargé")

Modèle spaCy chargé


In [2]:
import os, re
from collections import Counter

data_path = "./data/txt"  
files = sorted([f for f in os.listdir(data_path) if f.endswith(".txt")])

print("Nombre de fichiers :", len(files))
print("Exemples :", files[:5])

assert 500 <= len(files) <= 999, "Le sous-corpus doit contenir entre 500 et 999 documents."

Nombre de fichiers : 957
Exemples : ['KB_JB421_1950-01-04_01-00001.txt', 'KB_JB421_1950-01-04_01-00002.txt', 'KB_JB421_1950-01-04_01-00003.txt', 'KB_JB421_1950-01-04_01-00004.txt', 'KB_JB421_1950-01-04_01-00005.txt']


### 2. Construction d'un échantillon 

Ici je prends les 30 premiers fichiers et je les concatène pour faire un texte d’échantillon. Ça permet de tester rapidement les idées sans lancer des calculs lourds sur tout le corpus.

In [4]:
sample_files = files[:30]

sample_text = ""
for f in sample_files:
    with open(os.path.join(data_path, f), encoding="utf-8", errors="replace") as fh:
        sample_text += fh.read() + "\n"

print("Taille échantillon (caractères) :", len(sample_text))
print(sample_text[:800])

Taille échantillon (caractères) : 960933
JOURNAL QUOTIDIEN TELEPHONES : Abonnements et Publicité : 23. Direction et Rédaction : 670. C O. P. 7245. ABONNEMENTS : 1 an : 310 francs □ans tous ies Bureaux de postes. 42, rue des Déportés, Arlon ^ du Luxembourg Le n" 1,25 fr 56" ANNEE. — N" 3 Publicité : four toute pubh- rifé en dehors des provinces de Lu- cembourg «t de \amur ainsi qu* lu Grand-Duché ie Luxembourg, s'adr. à l'Office de Publicité, 36, r. Neuve, à Brucelles.  MERCREDI k Janvier 1950 Perspectives politiques de Tan neuf Objectifs immédiats ET TACHES DE DEMAIN L'ANNEE politique bel™ ne s'ou- <"» tbmH, qu'il s'agisse du secteur éeo- vrira effectivement aue le 10 nomique ou de la résorption du cho- janvier. C'est à cette date mage, que i. on évoque même le plan qu'est fixée la rentrée parle- international ou l'action dc M. V


### 3. Extraction des entités 

Dans cette partie, j’applique spaCy sur l’échantillon pour récupérer doc.ents. C’est ma base pour compter et voir ce qui revient le plus souvent (lieux, personnes, orga., etc.).

In [5]:
try:
    doc = nlp(sample_text)
    print("OK, doc créé ")
    print("Nb entités :", len(doc.ents))
except Exception as e:
    print("Erreur spaCy :", type(e)._name_, "-", e)

OK, doc créé 
Nb entités : 14388


In [7]:
from collections import Counter
ents = [(ent.text, ent.label_) for ent in doc.ents]

### 4. Top des entités 

Je compte les entités par texte et étiquette et j’affiche le Top-20. On retrouve surtout des lieux et quelques personnes/organisations

In [8]:
ent_counter = Counter(ents)
for (text, label), c in ent_counter.most_common(20):
    print(f"{text} ({label}) — {c}")

Bruxelles (LOC) — 95
Belgique (LOC) — 66
H (PER) — 65
M. (PER) — 57
Arlon (LOC) — 57
Allemagne (LOC) — 51
Gouvernement (LOC) — 47
Reuter (MISC) — 46
Paris (LOC) — 46
Etat (LOC) — 45
I (LOC) — 39
A. F. P. (PER) — 37
Etats-Unis (LOC) — 37
Antoine (PER) — 36
Liège (LOC) — 34
Roi (PER) — 32
Noël (MISC) — 31
Luxembourg (LOC) — 31
France (LOC) — 30
Ob (LOC) — 29


### 5. Mots-clés avec YAKE 

Pour cette partie je lance YAKE sur le même échantillon. On voit ressortir des formes fréquentes du discours. L’intérêt est de repérer vite les termes saillants pour guider la suite (sélection de thèmes, nettoyage, etc.).

In [9]:
import yake

kw_extractor = yake.KeywordExtractor(lan="fr", top=20)
keywords = kw_extractor.extract_keywords(sample_text)

keywords[:20]

[('nous', 0.00012664425725284438),
 ('nous nous', 0.00016912081833287546),
 ("d'un", 0.0003116998705537109),
 ('Janvier', 0.0003910968977894683),
 ('Bruxelles', 0.0004031977779913399),
 ("C'est", 0.00042647497633353874),
 ("d'une", 0.00047322419381982906),
 ("qu'il", 0.0005312615360993832),
 ('Nous avons', 0.0005440452643174053),
 ('Gouvernement', 0.0006473813170985569),
 ('ans', 0.0007267419768430703),
 ('Belgique', 0.0007889613436425659),
 ('grand', 0.0008402119974978227),
 ('pays', 0.0009171418767553618),
 ("Roi d'un Gouvernement", 0.0009527610680538197),
 ('cours', 0.0009916878408735073),
 ('Luxembourg', 0.001011681093443226),
 ('faire', 0.001082425773746076),
 ("cours d'une", 0.0010966605990488886),
 ('BELGE', 0.001103633395735381)]

### 6. Bigrams 

Je filtre les mots-clés pour garder que les bigrams (deux mots). Ces expressions aident à capter des tournures récurrentes et des micro-thèmes. 

In [10]:
bigrams = []
for kw, score in keywords:
    if len(kw.split()) == 2:
        bigrams.append((kw, score))

bigrams

[('nous nous', 0.00016912081833287546),
 ('Nous avons', 0.0005440452643174053),
 ("cours d'une", 0.0010966605990488886)]